In [2]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    SGDRegressor, SGDClassifier, LogisticRegression,
)
from sklearn.neural_network import MLPRegressor, MLPClassifier

from sklearn.metrics import (
    r2_score, mean_absolute_error, root_mean_squared_error,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
external_dataset=pd.read_excel(r"C:\Users\Sumit's\Downloads\archive (1)\External_Cibil_Dataset.xlsx")
internal_dataset=pd.read_excel(r"C:\Users\Sumit's\Downloads\archive (1)\Internal_Bank_Dataset.xlsx")

In [4]:
external_dataset.shape

(51336, 62)

In [5]:
internal_dataset.shape

(51336, 26)

In [6]:
df=pd.merge(
    external_dataset,
    internal_dataset,
    on="PROSPECTID",
    how="inner"
)
    

In [7]:
df.columns = [col.lower().strip() for col in df.columns]

In [8]:
print(df.shape)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.head()

(51336, 87)
Duplicate rows: 0


,prospectid,time_since_recent_payment,time_since_first_deliquency,time_since_recent_deliquency,num_times_delinquent,max_delinquency_level,max_recent_level_of_deliq,num_deliq_6mts,num_deliq_12mts,num_deliq_6_12mts,...,cc_tl,consumer_tl,gold_tl,home_tl,pl_tl,secured_tl,unsecured_tl,other_tl,age_oldest_tl,age_newest_tl
0,1,549,35,15,11,29,29,0,0,0,...,0,0,1,0,4,1,4,0,72,18
1,2,47,-99999,-99999,0,-99999,0,0,0,0,...,0,1,0,0,0,0,1,0,7,7
2,3,302,11,3,9,25,25,1,9,8,...,0,6,1,0,0,2,6,0,47,2
3,4,-99999,-99999,-99999,0,-99999,0,0,0,0,...,0,0,0,0,0,0,1,1,5,5
4,5,583,-99999,-99999,0,-99999,0,0,0,0,...,0,0,0,0,0,3,0,2,131,32


In [9]:
df["approved_flag"].value_counts()

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64

In [10]:
X = df.drop(columns=["approved_flag", "credit_score"])
y_linear = df["credit_score"].astype(float)
y_multiclass = df["approved_flag"].astype(str)

binary_map = {"P1": 1, "P2": 1, "P3": 0, "P4": 0}
y_binary = df["approved_flag"].map(binary_map)

assert y_binary.isna().sum() == 0, "approved_flag has values outside P1-P4"

print(y_multiclass.value_counts())
print(y_binary.value_counts())

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64
approved_flag
1    38002
0    13334
Name: count, dtype: int64


In [11]:
X_train, X_test, idx_train, idx_test = train_test_split(
    X, X.index, test_size=0.2, random_state=RANDOM_STATE
)

y_linear_train, y_linear_test = y_linear.loc[idx_train], y_linear.loc[idx_test]
y_multi_train, y_multi_test = y_multiclass.loc[idx_train], y_multiclass.loc[idx_test]
y_bin_train, y_bin_test = y_binary.loc[idx_train], y_binary.loc[idx_test]

print(X_train.shape, X_test.shape)

(41068, 85) (10268, 85)


In [23]:
# Empty list to store columns with high missing values
high_missing = []

# Check the specified columns one by one
for c in ["cc_utilization", "pl_utilization"]:
    
    # Check whether the column exists in X_train
    if c in X_train.columns:
        
        # Add the column to the high_missing list
        high_missing.append(c)

# Remove high-missing columns from training data
X_train = X_train.drop(columns=high_missing)

# Remove the same columns from testing data
X_test = X_test.drop(columns=high_missing)

In [24]:
# Columns where missing values will be filled using median
median_columns = [
    "age_oldest_tl",
    "age_newest_tl",
    "pct_currentbal_all_tl",
    "time_since_recent_payment"
]

# Process each column one by one
for col in median_columns:

    # Skip the column if it is not present in training data
    if col not in X_train.columns:
        continue

    # Replace -99999 with NaN (missing value)
    X_train[col] = X_train[col].replace(-99999, np.nan)
    X_test[col] = X_test[col].replace(-99999, np.nan)

    # Calculate median using training data only
    train_median = X_train[col].median()

    # Fill missing values in training data with training median
    X_train[col] = X_train[col].fillna(train_median)

    # Fill missing values in test data using the same training median
    X_test[col] = X_test[col].fillna(train_median)

In [ ]:
# List of delinquency-related columns
delinquency_columns = [
    "max_delinquency_level",
    "max_deliq_6mts",
    "max_deliq_12mts",
    "time_since_recent_delinquency",
    "time_since_first_delinquency",
]

# Keep only columns that are present in X_train
delinquency_columns = [
    c for c in delinquency_columns
    if c in X_train.columns
]

# Replace -99999 with 0 in training data
X_train[delinquency_columns] = X_train[delinquency_columns].replace(-99999, 0)

# Replace -99999 with 0 in testing data
X_test[delinquency_columns] = X_test[delinquency_columns].replace(-99999, 0)

In [ ]:
# List of enquiry-related columns
enquiry_columns = [
    "tot_enq", "cc_enq", "pl_enq", "cc_enq_16m", "cc_enq_112m",
    "pl_enq_16m", "pl_enq_112m", "enq_13m", "enq_16m", "enq_112m",
]

# Keep only columns that are present in X_train
enquiry_columns = [c for c in enquiry_columns if c in X_train.columns]

# Replace -99999 with 0 in training data
X_train[enquiry_columns] = X_train[enquiry_columns].replace(-99999, 0)

# Replace -99999 with 0 in testing data
X_test[enquiry_columns] = X_test[enquiry_columns].replace(-99999, 0)

In [16]:
col = "time_since_recent_enq"
if col in X_train.columns:
    train_mask = X_train[col] == -99999
    test_mask = X_test[col] == -99999
    fill_value = X_train.loc[~train_mask, col].max() + 1

    X_train["no_enquiry_flag"] = train_mask.astype(int)
    X_test["no_enquiry_flag"] = test_mask.astype(int)

    X_train[col] = X_train[col].replace(-99999, fill_value)
    X_test[col] = X_test[col].replace(-99999, fill_value)


In [25]:
col = "max_unsec_exposure_inpct"
if col in X_train.columns:
    X_train[col] = X_train[col].replace(-99999, 0)
    X_test[col] = X_test[col].replace(-99999, 0)

In [26]:
if "netmonthlyincome" in X_train.columns:
    X_train["netmonthlyincome_log"] = np.log1p(X_train["netmonthlyincome"].clip(lower=0))
    X_test["netmonthlyincome_log"] = np.log1p(X_test["netmonthlyincome"].clip(lower=0))
    X_train = X_train.drop(columns=["netmonthlyincome"])
    X_test = X_test.drop(columns=["netmonthlyincome"])

print("Remaining NaNs in X_train:", X_train.isna().sum().sum())
print("Remaining NaNs in X_test:", X_test.isna().sum().sum())

Remaining NaNs in X_train: 0
Remaining NaNs in X_test: 0


In [18]:
numeric_columns = X_train.select_dtypes(include=np.number).columns
categorical_columns = X_train.select_dtypes(include="object").columns
print(f"{len(numeric_columns)} numeric columns, {len(categorical_columns)} categorical columns")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed shapes:", X_train_processed.shape, X_test_processed.shape)

81 numeric columns, 5 categorical columns
Processed shapes: (41068, 104) (10268, 104)


In [19]:
X_all_processed = pd.concat([X_train_processed, X_test_processed], axis=0).sort_index()
y_linear_all = y_linear.loc[X_all_processed.index]
y_binary_all = y_binary.loc[X_all_processed.index]
y_multiclass_all = y_multiclass.loc[X_all_processed.index]

In [20]:

def evaluate_regression(model, X_te, y_te):
    y_pred = model.predict(X_te)
    return {
        "r2": r2_score(y_te, y_pred),
        "mae": mean_absolute_error(y_te, y_pred),
        "rmse": root_mean_squared_error(y_te, y_pred),
    }

def evaluate_classification(model, X_te, y_te, binary):
    y_pred = model.predict(X_te)
    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),
        "f1_macro": f1_score(y_te, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_te, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_te, y_pred, average="macro", zero_division=0),
        # Weighted averages
        "precision_weighted": precision_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_te, y_pred, average="weighted", zero_division=0
        ),

        # Confusion Matrix
        "confusion_matrix": confusion_matrix(y_te, y_pred),

        # Complete classification report
        "classification_report": classification_report(
            y_te,
            y_pred,
            zero_division=0,
            output_dict=True,
        ),
    }
    if binary and hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_te, y_prob)
    return metrics

In [21]:
N_FOLDS = 5
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
skfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

CV_RESULTS = []

REGRESSION_SCORING = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
}
CLASSIFICATION_SCORING = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
}
BINARY_ONLY_SCORING = {"roc_auc": "roc_auc"}

In [22]:
lin_model = LinearRegression()
start = time.time()
lin_model.fit(X_train_processed, y_linear_train)
fit_time = time.time() - start
print(evaluate_regression(lin_model, X_test_processed, y_linear_test))
print(fit_time)

{'r2': 0.9105891201623383, 'mae': 5.318019235628592, 'rmse': 6.115549937235495}
0.20722150802612305
